# Proposed validation — review before it runs

**What this measures:** The target metric is the ratio of CoIFNet's masked forecasting MAE to the impute-then-forecast baseline's MAE on the same 0.3-MCAR-corrupted ETTh1 lookback windows, computed with `pypots.nn.functional.calc_mae` exactly as `pypots/cli/evaluate.py` uses it — it directly operationalizes the claim's own wording ("at least as good as ... ideally better") as ratio ≤ 1.0, with no invented margin.

**Target metric:** `mae_ratio_coifnet_over_impute_baseline`

Remyx wrote this test for the change in this PR. **Nothing here has been executed** — there are no outputs, and no result is being claimed.

Edit it if the measurement is wrong, then mention `@remyx validate` again and it will run what you committed. If anything is missing at run time — an import, a dependency, a device — the run reports it and repairs what it can rather than failing silently.

This notebook is the only copy of the test: `.remyx/validation.yaml` points here, and the run executes these cells top to bottom.

In [ ]:
# papermill parameters — Remyx injects variant / ref / seed here
variant = ""
ref = ""
seed = 0

## Execution context

The cells below are the test Remyx drafted, split into cells; it was written as a script, so `__file__` is set to `eval/eval_coifnet_missing_forecast.py` (no such file is committed). This cell gives it what the command line would: its own path in `__file__`, an empty argument list so `argparse` sees no stray flags, and the papermill parameters as `REMYX_VARIANT` / `REMYX_REF` / `REMYX_SEED` for anything that wants them.

In [ ]:
import os, sys
ROOT = os.getcwd()  # the notebook runs with the repository root as its working directory
__file__ = os.path.join(ROOT, "eval/eval_coifnet_missing_forecast.py")
sys.argv = [__file__]
for _k in ("variant", "ref", "seed"):
    _v = globals().get(_k)
    if _v not in (None, ""):
        os.environ["REMYX_" + _k.upper()] = str(_v)
print("[remyx] cwd", ROOT, "| script", __file__)

In [ ]:
"""
Evaluation script: does CoIFNet's joint imputation-forecasting match/beat an
impute-then-forecast baseline (mean-imputation + DLinear) and stay competitive
with TimeMixer (trained on the same imputed data), on ETTh1 with ~30% injected
missing values in the lookback window?

Mirrors pypots/cli/evaluate.py's forecasting metrics (mse/mae via
pypots.nn.functional.calc_mse/calc_mae) and calls the real repo entry points
(CoIFNet.fit/forecast, DLinear.fit/forecast, TimeMixer.fit/forecast) -- no
reimplementation of the model logic.
"""

In [ ]:
import argparse
import hashlib
import inspect
import json
import os
import sys
import urllib.request

In [ ]:
import numpy as np
import torch

REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

In [ ]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

from pypots.forecasting import DLinear, TimeMixer  # noqa: E402
from pypots.nn.functional import calc_mae, calc_mse  # noqa: E402

In [ ]:
try:
    # This is exactly the symbol the PR diff adds to pypots/forecasting/__init__.py.
    from pypots.forecasting import CoIFNet

    COIFNET_AVAILABLE = True
except (ImportError, AttributeError):
    CoIFNet = None
    COIFNET_AVAILABLE = False

In [ ]:
def load_etth1(smoke: bool) -> np.ndarray:
    """Fetch ETTh1 (pinned public loader, cached) and return [T, 7] float32 array.

    Pinned to a specific immutable commit of the canonical ETDataset repo named in
    the user guidance (not the mutable "main" branch head), and the locally cached
    copy's integrity is verified against a sha256 checksum recorded next to it on
    first download -- if the cache is ever corrupted or silently swapped, re-running
    raises loudly instead of measuring against different bytes.
    """
    data_dir = os.path.join(os.path.dirname(__file__), "data")
    os.makedirs(data_dir, exist_ok=True)
    csv_path = os.path.join(data_dir, "ETTh1.csv")
    checksum_path = csv_path + ".sha256"
    # Pinned commit (immutable ref) of the canonical ETT benchmark repo, not "main".
    pinned_commit = "e51ba717249d5a2b45d9ecf428de7d3e78194ea6"
    url = (
        f"https://raw.githubusercontent.com/zhouhaoyi/ETDataset/"
        f"{pinned_commit}/ETT-small/ETTh1.csv"
    )
    if not os.path.exists(csv_path):
        urllib.request.urlretrieve(url, csv_path)

    with open(csv_path, "rb") as f:
        digest = hashlib.sha256(f.read()).hexdigest()
    if os.path.exists(checksum_path):
        with open(checksum_path, "r") as f:
            expected = f.read().strip()
        if digest != expected:
            raise RuntimeError(
                "ETTh1.csv checksum mismatch: the locally cached copy does not match "
                f"the checksum recorded for the pinned commit ({expected} vs {digest})."
            )
    else:
        with open(checksum_path, "w") as f:
            f.write(digest)

    n_rows_needed = 260 if smoke else 700
    rows = []
    with open(csv_path, "r") as f:
        f.readline()  # header
        for i, line in enumerate(f):
            if i >= n_rows_needed:
                break
            parts = line.strip().split(",")[1:]  # drop the 'date' column
            rows.append([float(x) for x in parts])
    return np.asarray(rows, dtype=np.float32)  # [T, 7]

In [ ]:
def make_windows(data: np.ndarray, n_steps: int, n_pred_steps: int, stride: int):
    T = data.shape[0]
    xs, ys = [], []
    for start in range(0, T - n_steps - n_pred_steps + 1, stride):
        xs.append(data[start:start + n_steps])
        ys.append(data[start + n_steps:start + n_steps + n_pred_steps])
    return np.stack(xs), np.stack(ys)

In [ ]:
def inject_missing(X: np.ndarray, rate: float, rng: np.random.Generator):
    keep = rng.random(X.shape) >= rate
    X_missing = X.copy()
    X_missing[~keep] = np.nan
    return X_missing

def mean_impute(X_with_nan: np.ndarray, fill_means: np.ndarray) -> np.ndarray:
    X = X_with_nan.copy()
    for c in range(X.shape[-1]):
        ch = X[..., c]
        ch[np.isnan(ch)] = fill_means[c]
    return X

In [ ]:
def build_kwargs(cls, dims: dict, epochs: int, batch_size: int) -> dict:
    """Fill in only the uniform BaseNNForecaster-style dims/training knobs that are
    actually present in the class's own __init__ signature -- discovered by
    introspection of the real class, never assumed."""
    sig = inspect.signature(cls.__init__)
    candidates = dict(
        n_steps=dims["n_steps"],
        n_features=dims["n_features"],
        n_pred_steps=dims["n_pred_steps"],
        n_pred_features=dims["n_pred_features"],
        epochs=epochs,
        batch_size=batch_size,
        device="cpu",
        verbose=False,
    )
    return {k: v for k, v in candidates.items() if k in sig.parameters}

In [ ]:
def train_and_forecast(model_cls, extra_kwargs, dims, epochs, batch_size,
                        train_X, train_Y, val_X, val_Y, test_X):
    kwargs = build_kwargs(model_cls, dims, epochs, batch_size)
    kwargs.update(extra_kwargs)
    model = model_cls(**kwargs)
    model.fit({"X": train_X, "X_pred": train_Y}, {"X": val_X, "X_pred": val_Y})
    return np.asarray(model.forecast({"X": test_X}))

In [ ]:
def mae_mse(pred: np.ndarray, target: np.ndarray):
    pred_t = torch.from_numpy(pred.astype(np.float32))
    target_t = torch.from_numpy(target.astype(np.float32))
    return float(calc_mae(pred_t, target_t, None)), float(calc_mse(pred_t, target_t, None))

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--variant", default=None)
    parser.add_argument("--ref", default=None)
    parser.add_argument("--seed", default=None)
    parser.parse_known_args()

    smoke = os.environ.get("REMYX_SMOKE", "0") == "1"

    n_steps = 24 if smoke else 48
    n_pred_steps = 6 if smoke else 12
    n_features = n_pred_features = 7
    stride = 6 if smoke else 4
    epochs = 1 if smoke else 5
    batch_size = 8 if smoke else 16
    missing_rate = 0.3

    raw = load_etth1(smoke)
    X_all, Y_all = make_windows(raw, n_steps, n_pred_steps, stride)
    n = X_all.shape[0]
    n_train = max(int(n * 0.6), 4)
    n_val = max(int(n * 0.2), 2)
    train_X, train_Y = X_all[:n_train], Y_all[:n_train]
    val_X, val_Y = X_all[n_train:n_train + n_val], Y_all[n_train:n_train + n_val]
    test_X, test_Y = X_all[n_train + n_val:], Y_all[n_train + n_val:]
    if test_X.shape[0] == 0:
        test_X, test_Y = val_X, val_Y

    rng = np.random.default_rng(SEED)
    train_X_missing = inject_missing(train_X, missing_rate, rng)
    val_X_missing = inject_missing(val_X, missing_rate, rng)
    test_X_missing = inject_missing(test_X, missing_rate, rng)

    fill_means = np.nan_to_num(np.nanmean(train_X_missing, axis=(0, 1)), nan=0.0)
    train_X_imp = mean_impute(train_X_missing, fill_means)
    val_X_imp = mean_impute(val_X_missing, fill_means)
    test_X_imp = mean_impute(test_X_missing, fill_means)

    dims = dict(n_steps=n_steps, n_features=n_features,
                n_pred_steps=n_pred_steps, n_pred_features=n_pred_features)

    # (a) impute-then-forecast baseline: mean imputation + DLinear
    baseline_pred = train_and_forecast(
        DLinear, {}, dims, epochs, batch_size,
        train_X_imp, train_Y, val_X_imp, val_Y, test_X_imp,
    )
    baseline_mae, baseline_mse = mae_mse(baseline_pred, test_Y)

    # (b) existing PyPOTS forecaster (TimeMixer) on the same imputed data
    tm_pred = train_and_forecast(
        TimeMixer, {}, dims, epochs, batch_size,
        train_X_imp, train_Y, val_X_imp, val_Y, test_X_imp,
    )
    timemixer_mae, timemixer_mse = mae_mse(tm_pred, test_Y)

    if COIFNET_AVAILABLE:
        # Feature explicitly enabled: joint imputation-forecasting on raw incomplete X.
        coif_extra = dict(
            use_reconstruct=True,
            use_mask=True,
            use_reversible_norm=True,
            use_revon=True,
            loss_lambda=0.1,
        )
        coif_sig = inspect.signature(CoIFNet.__init__)
        coif_extra = {k: v for k, v in coif_extra.items() if k in coif_sig.parameters}
        coif_pred_raw = train_and_forecast(
            CoIFNet, coif_extra, dims, epochs, batch_size,
            train_X_missing, train_Y, val_X_missing, val_Y, test_X_missing,
        ).astype(np.float64)
        frac_finite = float(np.isfinite(coif_pred_raw).mean())
        coif_pred_safe = np.nan_to_num(coif_pred_raw, nan=0.0, posinf=0.0, neginf=0.0)
        coifnet_mae, coifnet_mse = mae_mse(coif_pred_safe, test_Y)
    else:
        # Baseline/pre-change fallback: CoIFNet does not exist yet, so no CoIFNet
        # predictions are produced at all -- there is nothing to flag as non-finite,
        # so the guardrail is vacuously satisfied (1.0), and we emit degraded target
        # metrics so the *target* (not the guardrail) correctly reflects "no gain yet".
        frac_finite = 1.0
        coifnet_mae = baseline_mae * 10.0 + 1.0
        coifnet_mse = baseline_mse * 10.0 + 1.0

    eps = 1e-8
    metrics = {
        "mae_ratio_coifnet_over_impute_baseline": coifnet_mae / (baseline_mae + eps),
        "mse_ratio_coifnet_over_impute_baseline": coifnet_mse / (baseline_mse + eps),
        "mae_ratio_coifnet_over_timemixer": coifnet_mae / (timemixer_mae + eps),
        "frac_finite_predictions_coifnet": frac_finite,
        "coifnet_mae": coifnet_mae,
        "coifnet_mse": coifnet_mse,
        "impute_baseline_mae": baseline_mae,
        "impute_baseline_mse": baseline_mse,
        "timemixer_mae": timemixer_mae,
        "timemixer_mse": timemixer_mse,
    }
    print(json.dumps(metrics))

if __name__ == "__main__":
    main()

## The criteria this is judged against

From `.remyx/validation.yaml` — thresholds live here, not in the test, so a failing measurement reports rather than crashes.

```yaml
question:
  kind: capability
  ask: "On ETTh1 with ~30% injected missing values, does CoIFNet (joint imputation-forecasting) forecast at least as well as an impute-then-forecast baseline and competitively with TimeMixer trained on the same imputed data?"

loop: {max_iterations: 8, fix_code: true}

benchmarks:
  - name: coifnet-partial-obs-forecast
    suite: "eval/eval_coifnet_missing_forecast.py"
    scorer: mae_ratio_coifnet_over_impute_baseline
    baseline: none
    metrics:
      - name: mae_ratio_coifnet_over_impute_baseline
        role: target
        bar: goal
        direction: min
        threshold: 1.0
        reads_as: "CoIFNet's horizon MAE divided by the mean-imputation+DLinear baseline's MAE; <=1.0 means CoIFNet is at least as good."
      - name: frac_finite_predictions_coifnet
        role: guardrail
        bar: floor
        direction: max
        threshold: 1.0
        reads_as: "Fraction of CoIFNet's forecast entries that are finite (not NaN/Inf); RevON's observed-only stats must never blow up."
      - name: mse_ratio_coifnet_over_impute_baseline
        role: diagnostic
        bar: goal
        direction: min
        threshold: 1.0
        reads_as: "Same ratio as the target metric but on MSE, PyPOTS's other headline forecasting metric; reported for context, not gated (see policy)."
      - name: mae_ratio_coifnet_over_timemixer
        role: diagnostic
        bar: goal
        direction: min
        threshold: 1.0
        reads_as: "CoIFNet's MAE divided by TimeMixer-on-imputed-data's MAE; a harder bar than the target, reported not gated."
      - name: coifnet_mae
        role: diagnostic
        bar: ceiling
        direction: max
        threshold: 100.0
        reads_as: "CoIFNet's raw horizon MAE; ceiling is a loose sanity bound derived from ETTh1's raw (unnormalized) value range (roughly -20..60 across the 7 channels), so a well-behaved model's MAE must stay far below the channel range itself."
      - name: coifnet_mse
        role: diagnostic
        bar: ceiling
        direction: max
        threshold: 10000.0
        reads_as: "CoIFNet's raw horizon MSE; ceiling is the square-scale analog of the MAE sanity bound above (100^2), i.e. a model producing errors on the order of the raw value range, not a tighter fit."
      - name: impute_baseline_mae
        role: diagnostic
        bar: ceiling
        direction: max
        threshold: 100.0
        reads_as: "Mean-imputation + DLinear baseline's raw horizon MAE; same sanity ceiling as coifnet_mae."
      - name: impute_baseline_mse
        role: diagnostic
        bar: ceiling
        direction: max
        threshold: 10000.0
        reads_as: "Mean-imputation + DLinear baseline's raw horizon MSE; same sanity ceiling as coifnet_mse."
      - name: timemixer_mae
        role: diagnostic
        bar: ceiling
        direction: max
        threshold: 100.0
        reads_as: "TimeMixer-on-mean-imputed-data's raw horizon MAE; same sanity ceiling as coifnet_mae."
      - name: timemixer_mse
        role: diagnostic
        bar: ceiling
        direction: max
        threshold: 10000.0
        reads_as: "TimeMixer-on-mean-imputed-data's raw horizon MSE; same sanity ceiling as coifnet_mse."
    policy: {guardrail_veto: true}

compute: {tier: cpu}

held_constant:
  - "Same ETTh1 CSV, same sliding-window split (train/val/test), and the same seeded 0.3 MCAR missingness mask applied to the lookback window X for all three pipelines"
  - "Same n_steps/n_pred_steps/n_features/n_pred_features, epochs, batch_size and seed across CoIFNet, the impute-then-forecast baseline, and TimeMixer"
  - "X_pred (forecast horizon ground truth) stays fully observed for all pipelines so MAE/MSE are computed on identical targets with no masking"

avoid:
  - "Do not treat this run as reproducing the paper's 24.40%/23.81% figure, which is reported at missing rate 0.6 against an unnamed SOTA comparator, not at 0.3 against the weaker impute-then-forecast baseline required here"
  - "Do not use wall-clock timing as the instrument; only masked MAE/MSE on the horizon are measured"
  - "Do not swap in PhysioNet silently if ETT is reachable; PhysioNet has no forecasting-horizon analog in the paper and would only be a repo-internal sanity check"

report:
  headline: "CoIFNet's joint imputation-forecasting matches or beats a mean-imputation+DLinear baseline on partially-observed ETTh1, with TimeMixer reported alongside for context."
  findings:
    - "CoIFNet's MAE is {mae_ratio_coifnet_over_impute_baseline:%} of the impute-then-forecast baseline's MAE (<=100% means at least as good)."
    - "CoIFNet's MSE is {mse_ratio_coifnet_over_impute_baseline:%} of the baseline's MSE."
    - "CoIFNet's MAE is {mae_ratio_coifnet_over_timemixer:%} of TimeMixer-on-imputed-data's MAE."
    - "{frac_finite_predictions_coifnet:%} of CoIFNet's forecast entries are finite."
    - "Raw MAE/MSE side by side: CoIFNet {coifnet_mae}/{coifnet_mse}, impute+DLinear baseline {impute_baseline_mae}/{impute_baseline_mse}, TimeMixer-on-imputed {timemixer_mae}/{timemixer_mse}."
  establishes: ["Whether CoIFNet's joint imputation-forecasting objective, trained end-to-end on 30%-missing ETTh1 input, produces forecast error at or below a standard impute-then-forecast pipeline on the same data."]
  does_not_establish: ["The paper's reported 24.40%/23.81% improvement over an unnamed SOTA method at missing rate 0.6 (a different rate, a different, stronger comparator, and a different benchmark suite than run here)", "Memory (4.3x) or time (2.1x) efficiency claims, which are not measured by this harness"]
  not_measured: ["Point vs block missingness pattern sensitivity", "Behavior at the paper's 0.6 missing rate", "Multiple random seeds / variance across runs"]
  caveat: "This repo has no BenchPOTS apparatus (its own VALIDATION.md defers that parity run to a human/GPU step), so this harness performs its own three-way comparison at 0.3 missing rate rather than replicating the paper's 0.6-missing-rate SOTA-relative number."
  next: "Run the BenchPOTS-parity protocol sketched in pypots/forecasting/coifnet/VALIDATION.md (point + block missingness at {0.1,0.3,0.6}, multiple seeds, against CSDI/TimeMixer) on a GPU to check the paper's 24% figure directly."

provenance:
  mae_ratio_coifnet_over_impute_baseline: "user_guidance"
  mse_ratio_coifnet_over_impute_baseline: "user_guidance"
  mae_ratio_coifnet_over_timemixer: "user_guidance"
  frac_finite_predictions_coifnet: "protocol_doc:pypots/forecasting/coifnet/VALIDATION.md"
  held_constant: "protocol_doc:pypots/forecasting/coifnet/VALIDATION.md"
  suite: "synthesized"
  baseline: "claim_analysis:mismatches (no BenchPOTS apparatus in-repo, capability question, no baseline arm)"
```